In [26]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [27]:
groups = train['layout_id']

In [28]:
drop_cols = ['ID', 'layout_id', 'scenario_id', 'avg_delay_minutes_next_30m']
X = train.drop(drop_cols, axis=1)
y = train['avg_delay_minutes_next_30m']


In [29]:
best_params = {
    'n_estimators': 1000,
    'learning_rate': 0.0100,
    'num_leaves': 140,
    'max_depth': 8,
    'min_child_samples': 25,
    'subsample': 0.9608,
    'colsample_bytree': 0.6607,
    'reg_alpha': 1.2338,
    'reg_lambda': 7.6823,
    'random_state': 42,
    'verbose': -1
}

In [31]:
import optuna
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

def objective_xgb(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'random_state': 42,
    }

    gkf = GroupKFold(n_splits=5)
    maes = []
    for tr_idx, va_idx in gkf.split(X, y, groups=groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        m = XGBRegressor(**params)
        m.fit(X_tr, np.log1p(y_tr))
        pred = np.expm1(m.predict(X_va))
        maes.append(mean_absolute_error(y_va, pred))
    return np.mean(maes)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)

print('XGB 최적 파라미터:', study_xgb.best_params)
print('XGB 최적 MAE:', study_xgb.best_value)

[I 2026-06-29 16:03:08,453] A new study created in memory with name: no-name-3843c52a-6e13-4112-a1ec-6a056d2fcc52
[I 2026-06-29 16:05:53,748] Trial 0 finished with value: 9.530763711194833 and parameters: {'learning_rate': 0.03174822760169612, 'max_depth': 11, 'min_child_weight': 16, 'subsample': 0.600992813435277, 'colsample_bytree': 0.6675492439436362, 'reg_alpha': 2.7282752302381033, 'reg_lambda': 7.323209311792562}. Best is trial 0 with value: 9.530763711194833.
[I 2026-06-29 16:06:36,784] Trial 1 finished with value: 9.444172598608553 and parameters: {'learning_rate': 0.012336488907926107, 'max_depth': 6, 'min_child_weight': 15, 'subsample': 0.7670804104821721, 'colsample_bytree': 0.8409236457443928, 'reg_alpha': 0.5381695537389097, 'reg_lambda': 6.2995285134694825}. Best is trial 1 with value: 9.444172598608553.
[I 2026-06-29 16:07:03,745] Trial 2 finished with value: 9.550912172002985 and parameters: {'learning_rate': 0.07440585100217323, 'max_depth': 4, 'min_child_weight': 19, 

XGB 최적 파라미터: {'learning_rate': 0.010056519402939588, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.6586515826835516, 'colsample_bytree': 0.8854251683912562, 'reg_alpha': 3.3611535023493744, 'reg_lambda': 2.6706537592602744}
XGB 최적 MAE: 9.43557378489253


In [32]:
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

lgb_params = {  
    'n_estimators': 1000, 'learning_rate': 0.0100, 'num_leaves': 140,
    'max_depth': 8, 'min_child_samples': 25, 'subsample': 0.9608,
    'colsample_bytree': 0.6607, 'reg_alpha': 1.2338, 'reg_lambda': 7.6823,
    'random_state': 42, 'verbose': -1
}
xgb_params = {  # 방금 XGB 튜닝 결과
    'n_estimators': 1000, 'learning_rate': 0.0101, 'max_depth': 7,
    'min_child_weight': 3, 'subsample': 0.6587, 'colsample_bytree': 0.8854,
    'reg_alpha': 3.3612, 'reg_lambda': 2.6707, 'random_state': 42
}

gkf = GroupKFold(n_splits=5)
ens_maes = []

for tr_idx, va_idx in gkf.split(X, y, groups=groups):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    y_tr_log = np.log1p(y_tr)

    m_lgb = LGBMRegressor(**lgb_params); m_lgb.fit(X_tr, y_tr_log)
    m_xgb = XGBRegressor(**xgb_params); m_xgb.fit(X_tr, y_tr_log)

    p_lgb = np.expm1(m_lgb.predict(X_va))
    p_xgb = np.expm1(m_xgb.predict(X_va))
    p_ens = (p_lgb + p_xgb) / 2  # 50:50 평균

    ens_maes.append(mean_absolute_error(y_va, p_ens))

print(f'앙상블 MAE: {np.mean(ens_maes):.3f}')
print(f'(비교) LGB 단독: 9.431')

앙상블 MAE: 9.427
(비교) LGB 단독: 9.431


In [33]:
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

gkf = GroupKFold(n_splits=5)
ens_maes = []

for tr_idx, va_idx in gkf.split(X, y, groups=groups):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    y_tr_log = np.log1p(y_tr)

    # 3개 모델
    m_lgb = LGBMRegressor(**lgb_params); m_lgb.fit(X_tr, y_tr_log)
    m_xgb = XGBRegressor(**xgb_params); m_xgb.fit(X_tr, y_tr_log)
    m_rf = RandomForestRegressor(n_estimators=200, max_depth=12, 
                                  n_jobs=-1, random_state=42)
    m_rf.fit(X_tr, y_tr_log)

    p_lgb = np.expm1(m_lgb.predict(X_va))
    p_xgb = np.expm1(m_xgb.predict(X_va))
    p_rf = np.expm1(m_rf.predict(X_va))

    # 3개 평균
    p_ens = (p_lgb + p_xgb + p_rf) / 3

    ens_maes.append(mean_absolute_error(y_va, p_ens))

print(f'3-모델 앙상블 MAE: {np.mean(ens_maes):.3f}')
print(f'(비교) LGB 단독: 9.431 | 2-모델 앙상블: 9.427')

3-모델 앙상블 MAE: 9.420
(비교) LGB 단독: 9.431 | 2-모델 앙상블: 9.427
